# Cloud Storage (GCS) Hands-On Lab — Advanced Python SDK Capabilities
### GCP Training Program — Day 1, Module 1: Storage & Ingestion

**What this notebook covers:** this is a companion to `GCS_Hands_On_Lab_Day1.ipynb`, which walked
through the `google-cloud-storage` client library one operation at a time. This notebook stays on
that same SDK — **no CLI, no subprocess, no shelling out anywhere** — and goes one level deeper into
what the Python client can do when you're writing real automation rather than one-off demo cells:

- **Retry & error handling** — catching typed exceptions (`NotFound`, `Conflict`, `Forbidden`) and
  configuring custom retry policies for transient failures.
- **Batch requests** — sending many small metadata operations (updates, deletes) over a *single*
  HTTP batch instead of one round trip per object.
- **Transfer Manager** — parallel multi-file uploads/downloads, and parallel chunked transfer of a
  single large file, with a timing comparison against sequential transfer so the benefit is visible,
  not just claimed.

**Caveat for the class:** Transfer Manager is a newer addition to `google-cloud-storage` (available
from v2.7.0 onward, with API refinements in later minor versions). Pin or check your installed
version if something here doesn't match what you see in the official docs.

**This notebook is fully self-contained.** Just run the cells top to bottom:
1. The **Authenticate** cell will pop up a Google sign-in flow.
2. The **Configure** cell will *ask you* for your Project ID, region, and bucket name.
3. Everything after that runs against your own project automatically.

**Prerequisites (mention to the class before running):**
1. A GCP project with billing enabled.
2. IAM role on the project: `roles/storage.admin` (or equivalent) for the account you sign in with.
3. Python package `google-cloud-storage` (installed in a cell below).


## 1. Authenticate This Session
Sign in with the Google account that has access to your GCP project.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth application-default login")

## 2. Install & Import the Client Library
The advanced features used in this notebook (`transfer_manager`, typed exceptions, `Retry` policies)
all live inside `google-cloud-storage` — no extra packages needed.

In [ ]:
!pip install --quiet "google-cloud-storage>=2.10.0"

In [ ]:
import time
import os
from google.cloud import storage
from google.cloud.storage import transfer_manager
from google.api_core import exceptions as gcs_exceptions
from google.api_core.retry import Retry

print("google-cloud-storage imported successfully")

## 3. Configure Your Project, Region & Bucket
Enter your own values when prompted below — nothing to hand-edit in code. The bucket name gets a
timestamp suffix so it's globally unique (bucket names are global across all of GCS).

In [ ]:
PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your region [default: asia-south1]: ").strip()
if not REGION:
    REGION = "asia-south1"

_default_bucket = f"gcs-advanced-sdk-demo-{int(time.time())}"
BUCKET_NAME = input(f"Enter a bucket name [default: {_default_bucket}]: ").strip()
if not BUCKET_NAME:
    BUCKET_NAME = _default_bucket

client = storage.Client(project=PROJECT_ID)
print("Client initialized for project:", client.project)
print("Target bucket name:", BUCKET_NAME)

## 4. Quick Bucket Setup
Basic bucket creation was covered in depth in the first SDK lab — just enough here to have
something to run the advanced features against.

In [ ]:
bucket = client.bucket(BUCKET_NAME)
bucket.storage_class = "STANDARD"
bucket = client.create_bucket(bucket, location=REGION)
print(f"Created gs://{bucket.name} in {bucket.location}")

## 5. Retry & Error Handling
### 5.1 Catching Typed Exceptions
Every Storage SDK call can fail for reasons a script needs to branch on — the object doesn't exist,
the bucket already exists, or the caller lacks permission. The SDK raises typed exceptions from
`google.api_core.exceptions` for each of these instead of one generic error, so you can handle each
case differently.

In [ ]:
# 404 — object does not exist
try:
    client.bucket(BUCKET_NAME).blob("does-not-exist.txt").download_as_text()
except gcs_exceptions.NotFound as e:
    print("Caught NotFound as expected:", e.message if hasattr(e, "message") else str(e))

# 409 — bucket name already taken (globally unique names — this one almost certainly already exists)
try:
    client.create_bucket("gcs-training-demo-bucket-name-already-taken")
except gcs_exceptions.Conflict as e:
    print("Caught Conflict as expected:", e.message if hasattr(e, "message") else str(e))

### 5.2 Custom Retry Policies for Transient Failures
The client library already retries idempotent operations (like downloads) automatically on
transient errors (503s, connection resets). For non-idempotent operations, or to tune how
aggressively the client retries, pass a custom `Retry` object explicitly.

In [ ]:
# A retry policy: retry on server errors and rate limiting, back off up to 30s, give up after 60s total
custom_retry = Retry(
    predicate=Retry.if_exception_type(
        gcs_exceptions.ServiceUnavailable,
        gcs_exceptions.TooManyRequests,
        gcs_exceptions.InternalServerError,
    ),
    initial=1.0,
    maximum=30.0,
    multiplier=2.0,
    deadline=60.0,
)

blob = bucket.blob("uploads/retry_demo.txt")
blob.upload_from_string("Uploaded with a custom retry policy applied.", retry=custom_retry)
print("Uploaded uploads/retry_demo.txt with a custom retry policy.")

## 6. Batch Requests
### 6.1 Create Several Small Objects to Batch Against

In [ ]:
for i in range(5):
    bucket.blob(f"batch-demo/file_{i}.txt").upload_from_string(f"content of file {i}")
print("Created 5 objects under batch-demo/")

### 6.2 Batch-Update Metadata on All 5 Objects in One HTTP Batch
Without batching, updating metadata on 5 objects is 5 separate HTTP requests. `client.batch()` groups
them into a single batch request — the more objects, the bigger the savings in round-trip latency.

**Caveat:** batching applies to metadata-style calls (patch/update/delete), not to upload/download of
object *content* — those go through Transfer Manager instead (Section 7).

In [ ]:
blobs = [bucket.blob(f"batch-demo/file_{i}.txt") for i in range(5)]

with client.batch():
    for b in blobs:
        b.metadata = {"batch-tagged": "true"}
        b.patch()

# Verify — re-fetch one object and confirm the metadata landed
check = bucket.blob("batch-demo/file_0.txt")
check.reload()
print("Metadata after batch update:", check.metadata)

### 6.3 Batch-Delete All 5 Objects in One HTTP Batch

In [ ]:
with client.batch():
    for b in blobs:
        b.delete()

print("Deleted all 5 batch-demo objects in a single batch request.")

## 7. Transfer Manager — Parallel Multi-File Uploads
### 7.1 Create Sample Local Files
A handful of small files stand in for a real batch of files (images, logs, exports) you'd want to
upload together.

In [ ]:
os.makedirs("transfer_demo", exist_ok=True)
filenames = []
for i in range(10):
    fname = f"transfer_demo/part_{i:02d}.txt"
    with open(fname, "w") as f:
        f.write(f"sample payload for file {i}\n" * 1000)
    filenames.append(fname)

print(f"Created {len(filenames)} local sample files.")

### 7.2 Baseline: Sequential Upload (One Blob at a Time)
Time this first so the parallel version below has something concrete to beat.

In [ ]:
start = time.time()
for fname in filenames:
    blob = bucket.blob(f"transfer-demo/{os.path.basename(fname)}")
    blob.upload_from_filename(fname)
sequential_seconds = time.time() - start

print(f"Sequential upload of {len(filenames)} files took {sequential_seconds:.2f}s")

# Clean up before the parallel run so we're comparing a true apples-to-apples upload
with client.batch():
    for fname in filenames:
        bucket.blob(f"transfer-demo/{os.path.basename(fname)}").delete()

### 7.3 Parallel Upload With Transfer Manager
`upload_many_from_filenames` uploads a whole list of local files concurrently using a thread pool —
one call instead of a manual loop, and materially faster for many small-to-medium files.

In [ ]:
start = time.time()
results = transfer_manager.upload_many_from_filenames(
    bucket,
    filenames,
    source_directory="",
    blob_name_prefix="transfer-demo/",
    max_workers=8,
)
parallel_seconds = time.time() - start

for fname, result in zip(filenames, results):
    status = "OK" if result is None else f"FAILED: {result}"
    print(f"{fname}: {status}")

print(f"\nParallel upload of {len(filenames)} files took {parallel_seconds:.2f}s")
print(f"Sequential took {sequential_seconds:.2f}s — parallel was {sequential_seconds / parallel_seconds:.1f}x faster")

## 8. Transfer Manager — Parallel Multi-File Downloads
The download counterpart to Section 7 — pull every object under a prefix down to a local directory
concurrently instead of looping one `download_to_filename()` call at a time.

In [ ]:
blob_names = [f"transfer-demo/{os.path.basename(fname)}" for fname in filenames]

os.makedirs("transfer_demo_downloaded", exist_ok=True)
start = time.time()
results = transfer_manager.download_many_to_path(
    bucket,
    blob_names,
    destination_directory="transfer_demo_downloaded/",
    max_workers=8,
)
download_seconds = time.time() - start

for name, result in zip(blob_names, results):
    status = "OK" if result is None else f"FAILED: {result}"
    print(f"{name}: {status}")

print(f"\nParallel download of {len(blob_names)} files took {download_seconds:.2f}s")

## 9. Transfer Manager — Chunked Parallel Transfer of One Large File
Sections 7-8 parallelized *across* many files. This section parallelizes *within* a single large
file — Transfer Manager splits it into chunks, uploads/downloads them concurrently, then reassembles.
Useful for large exports, model checkpoints, or backup archives.

**Cost/time note:** this cell generates a ~50MB local file to have something worth chunking — adjust
`size_mb` down if you're on a slow connection in class.

In [ ]:
size_mb = 50
large_file = "transfer_demo/large_file.bin"
with open(large_file, "wb") as f:
    f.write(os.urandom(1024 * 1024) * size_mb)

print(f"Created a {size_mb}MB local file: {large_file}")

In [ ]:
large_blob_name = "transfer-demo/large_file.bin"

start = time.time()
transfer_manager.upload_chunks_concurrently(
    large_file,
    bucket.blob(large_blob_name),
    chunk_size=8 * 1024 * 1024,  # 8MB chunks
    max_workers=4,
)
chunked_upload_seconds = time.time() - start
print(f"Chunked parallel upload of {size_mb}MB took {chunked_upload_seconds:.2f}s")

In [ ]:
downloaded_large_file = "transfer_demo_downloaded/large_file_downloaded.bin"

start = time.time()
transfer_manager.download_chunks_concurrently(
    bucket.blob(large_blob_name),
    downloaded_large_file,
    chunk_size=8 * 1024 * 1024,
    max_workers=4,
)
chunked_download_seconds = time.time() - start
print(f"Chunked parallel download of {size_mb}MB took {chunked_download_seconds:.2f}s")

assert os.path.getsize(downloaded_large_file) == os.path.getsize(large_file), "Downloaded size mismatch!"
print("Verified: downloaded file size matches the original.")

## 10. Cleanup — Delete All Objects & the Bucket
Run this **last**, once every demo above has been shown, so the class sees teardown too.

In [ ]:
# Delete every live object AND every archived version
for b in list(client.list_blobs(bucket, versions=True)):
    bucket.blob(b.name, generation=b.generation).delete()

bucket.delete()
print(f"Bucket {BUCKET_NAME} and all contents deleted.")

# Clean up local scratch files
import shutil
shutil.rmtree("transfer_demo", ignore_errors=True)
shutil.rmtree("transfer_demo_downloaded", ignore_errors=True)